<a href="https://colab.research.google.com/github/A01801224/TC3009C-RI/blob/main/Reto_Primer_Avance_Spaceship_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Después agregamos portada y demás pero por ahora hago este checklist de lo que proponemos poner y como vamos:

EDA (.shape, .info, .describe, .isnull().sum(), variable objetivo, .head(), .sample(), .tail(), .dtypes, .nunique() )

Datos nulos

Tipos de dato y cardinalidad

Variables categóricas ( .value_counts(), .groupby(), .mean() )

Tendencia central y dispersión

Skewness

Outliers-Regla de Tukey

Correlación

Tabla de contingencia ( pd.crosstab() )

Boxplots

Variables numéricas + interpretación de skewness

Matriz de correlación

In [4]:
import pandas as pd

# Pega aquí el enlace Raw de tu archivo
url_test = "https://raw.githubusercontent.com/A01801224/TC3009C-RI/refs/heads/main/test.csv"
url_train = "https://raw.githubusercontent.com/A01801224/TC3009C-RI/refs/heads/main/train.csv"

# Pandas lee el archivo directamente desde GitHub
df_test = pd.read_csv(url_test)
df_train = pd.read_csv(url_train)


# EDA

In [5]:
#Revisión de train

df_train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [6]:
df_train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [7]:
df_train.isnull().sum()

,0
PassengerId,0
HomePlanet,201
CryoSleep,217
Cabin,199
Destination,182
Age,179
VIP,203
RoomService,181
FoodCourt,183
ShoppingMall,208


In [8]:
#Calculo de mean para que las columnas con valores binarios
(df_train.isnull().mean()*100).round(4).sort_values(ascending=False)

,0
CryoSleep,2.4963
ShoppingMall,2.3927
VIP,2.3352
HomePlanet,2.3122
Name,2.3007
Cabin,2.2892
VRDeck,2.1627
Spa,2.1051
FoodCourt,2.1051
Destination,2.0936


Al hacer el calculo de porcentaje de valores nulos por columna, observamos que casi todas las variables (a excepción ed PassengerId y Transported) presentan un porcentaje de datos faltante muy similar, entre 2.05% y 2.50%. No hay ninguna columna que destaque por encima de las demás

## Primer Acercamineto, Analisis y Clasificación de Variables

In [9]:
df_train.nunique()

,0
PassengerId,8693
HomePlanet,3
CryoSleep,2
Cabin,6560
Destination,3
Age,80
VIP,2
RoomService,1273
FoodCourt,1507
ShoppingMall,1115


Este primer acercamineto, analisis y clasificación de variables esta sujeto a cambios

**Categóricas** (baja cardinalidad, útiles para value_counts/groupby directo):
- HomePlanet
- CryoSleep
- Destination
- VIP
- Transported (variable objetivo)

**Numéricas continuas** (tiene sentido calcular media, mediana, dispersión):
- Age
- RoomService
- FoodCourt
- ShoppingMall
- Spa
- VRDeck

**Identificadores ** (no son categóricas ni numéricas puras tal como están):
- PassengerId → posible extracción de número de grupo
- Cabin → posible división en Deck, Number, Side
- Name → no aporta directamente al análisis en su forma actual

## Analisis Especifico de Variables

**Revision Passenger**

Es importante mencionar que el formato de `PassengerId` es `gggg_pp`,
donde `gggg` es un identificador de grupo de 4 dígitos y `pp` es el número
de pasajero dentro de ese grupo (dos dígitos). Esta información proviene
del diccionario de datos de la competencia de Kaggle.

En este contexto, un grupo no necesariamente significa una familia, aunque
la mayoría de estos sí lo son. Esto es relevante porque nos permite explorar
si viajar en grupo (y no solo el vínculo familiar en sí) tiene alguna relación
con la probabilidad de haber sido transportado — por ejemplo, si los
integrantes de un mismo grupo tienden a compartir el mismo resultado en
`Transported`.

In [21]:
#Extraer el grpo de passengerid
df_train["group"] = df_train["PassengerId"].str.split("_").str[0]
print(df_train["group"])

0       0001
1       0002
2       0003
3       0003
4       0004
        ... 
8688    9276
8689    9278
8690    9279
8691    9280
8692    9280
Name: group, Length: 8693, dtype: object


Extraemos el identificador de grupo (`gggg`) de cada pasajero a partir de
`PassengerId`, dividiendo el valor por el separador `_` y quedándonos con
la primera parte. Los valores resultantes (0001, 0002, 0003...) corresponden
al grupo, no al número de pasajero dentro del grupo (`pp`).

In [22]:
group_sizes = df_train["group"].value_counts()
print(group_sizes)

group
9081    8
4005    8
8988    8
5133    8
4256    8
       ..
0022    1
0016    1
0015    1
0014    1
0012    1
Name: count, Length: 6217, dtype: int64


In [23]:
group_size_counts = group_sizes.value_counts().sort_index()
print(group_size_counts)

count
1    4805
2     841
3     340
4     103
5      53
6      29
7      33
8      13
Name: count, dtype: int64


Podemso observar que lso grupos más grandes son de 8 personas mientras que el mas pequeño es de solamente una. Con un total de 6217 grupos distintos entre los 8693 pasajeros totales.

También podemos observar que 4805 de esos grupos tienen un tamaño de 1, lo que siginifica que aproximadamnet el 55% de los pasajeros viajan sin acompáñante. El resto se distribuye en grupos cada vez más pequeños conforme aumenta el tamaño, desde 841 grupos de 2 personas hasta 13 grupos de 8 personas.  

**Revisión de HomePlanet**

In [11]:
df_train["HomePlanet"].unique()

array(['Europa', 'Earth', 'Mars', nan], dtype=object)

Home planet nos indica el planeta desde donde partió el pasajero, en este caso Europa, Earth y Mars. Podemos ver que hay valores nulos en esta columna

In [12]:
df_train["HomePlanet"].value_counts()

,count
HomePlanet,
Earth,4602
Europa,2131
Mars,1759


In [24]:
df_train["HomePlanet"].isnull().sum()

np.int64(201)

In [27]:
df_train["HomePlanet"].value_counts(normalize=True, dropna=False) * 100

,proportion
HomePlanet,
Earth,52.939146
Europa,24.513977
Mars,20.234672
NaN,2.312205


Podemos obseravr que al rededor del 53% de pasajeros vienen de la tierra, el 25% vienen de Europa, el 20% viene de Marte y el resto son valores nulos lo que quiere decir que es información que se perdió en el incidente